# Aula 05 — Processamento de Linguagem Natural

Células na ordem dos slides; algumas dependem de variáveis de células anteriores.

## Slide 12 — SUAVIZAÇÃO DE LAPLACE EVITA PROBABILIDADES ZERO

*Bloco 01*

In [1]:
# Suavização de Laplace (add-one smoothing)
# Adiciona alpha (default=1.0) ao numerador,
# e alpha * |vocab| ao denominador
# Antes da suavização:
# P('quebrado' | positivo) = 0 / 1000 = 0  ← problema
# Depois da suavização (alpha=1):
# P('quebrado' | positivo) = (0 + 1) / (1000 + |vocab|)
#                          = 1 / 31000
#                          ≈ 0.000032  ← pequeno mas não zero
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB(alpha=1.0)  # alpha = força da suavização

## Dados de exemplo (adicionado no notebook)

Os slides seguintes usam `reviews` e `labels`, que não aparecem nos slides.

In [ ]:
# [notebook] Dados sintéticos de exemplo — os slides assumem um corpus "já preparado" (reviews + labels)
import random
random.seed(42)
_pos = ["Adorei o produto", "Excelente qualidade", "Entrega rápida e bem embalado", "Recomendo muito",
        "Funciona perfeitamente", "Ótimo atendimento", "Superou minhas expectativas", "Produto maravilhoso"]
_neg = ["Péssimo produto", "Chegou quebrado", "Entrega atrasada", "Não recomendo",
        "Veio com defeito", "Atendimento horrível", "Qualidade ruim", "Muito decepcionado"]
_neu = ["comprei na semana passada", "o preço estava bom", "a embalagem era simples", "usei por alguns dias",
        "chegou em casa ontem", "a cor é como na foto"]
reviews_corpus, labels = [], []
for _ in range(400):
    y_ = random.randint(0, 1)
    frases = random.sample(_pos if y_ else _neg, 2) + random.sample(_neu, 1)
    random.shuffle(frases)
    reviews_corpus.append(". ".join(frases) + ".")
    labels.append(y_)
reviews = reviews_corpus  # nome usado na aula 05
print(len(reviews_corpus), "reviews | positivas:", sum(labels))

## Slide 14 — MULTINOMIAL NB + TF-IDF IMPLEMENTAÇÃO EM PIPELINE

*Bloco 02*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
# Hipotético: 'reviews' é lista de strings, 'labels' é lista 0/1
X_train, X_test, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.2, stratify=labels, random_state=42
)
# Pipeline = vetorização + classificador em UM ÚNICO objeto
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=5, sublinear_tf=True)),
    ('clf', MultinomialNB(alpha=1.0)),
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['negativo', 'positivo']))

## Slide 15 — SAÍDA DO CLASSIFICATION_REPORT MÉTRICAS POR CLASSE

*Bloco 03*

In [ ]:
# Saída ilustrativa do classification_report:
#               precision    recall  f1-score   support
#     negativo       0.85      0.82      0.83      1000
#     positivo       0.83      0.86      0.84      1000
#     accuracy                           0.84      2000
#    macro avg       0.84      0.84      0.84      2000
# weighted avg       0.84      0.84      0.84      2000
# Como ler:
# - precision: das previsões positivas, quantas certas?
# - recall: dos positivos reais, quantos encontramos?
# - f1-score: harmônica de precision e recall
# - support: número de exemplos por classe
# - macro avg: média simples (todas classes pesam igual)
# - weighted avg: ponderada pelo support

## Slide 17 — REGRESSÃO LOGÍSTICA O CAVALO-DE-BATALHA DA CLASSIFICAÇÃO

*Bloco 04*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(
        max_iter=1000,
        solver='liblinear',     # rápido para texto esparso
        C=1.0,                  # regularização (menor C = mais regularização)
        class_weight='balanced'  # ajusta para classes desbalanceadas
    )),
])
pipe.fit(X_train, y_train)

## Slide 18 — INTERPRETABILIDADE DOS COEFICIENTES A GRANDE VANTAGEM DA LR

*Bloco 05*

In [ ]:
# Acessando coeficientes do modelo treinado
import numpy as np
vocab = pipe.named_steps['tfidf'].get_feature_names_out()
coefs = pipe.named_steps['clf'].coef_[0]
# Top 10 palavras mais POSITIVAS (coef alto)
top_pos = np.argsort(coefs)[-10:][::-1]
print('Mais associadas a POSITIVO:')
for i in top_pos:
    print(f'  {vocab[i]:<20s} ({coefs[i]:+.3f})')
# Saída:
#   adorei         (+2.450)
#   excelente      (+2.103)
#   maravilhoso    (+1.987)
#   recomendo      (+1.812)
#   perfeito       (+1.654)
# Top 10 NEGATIVAS
top_neg = np.argsort(coefs)[:10]
print('Mais associadas a NEGATIVO:')
# pessimo, ruim, quebrado, defeituoso, horrivel, ...

## Slide 29 — CLASS_WEIGHT='BALANCED' LIDANDO COM CLASSES DESBALANCEADAS

*Bloco 06*

In [ ]:
# Sem class_weight: modelo prevê tudo como majoritária
LogisticRegression()
# Recall na classe minoritária: ~5%  ← inútil
# Com class_weight='balanced': pesos inversos à frequência
LogisticRegression(class_weight='balanced')
# Recall na classe minoritária: ~75%  ← funcional!
# Equivalente:
# class_weight={0: 1.0, 1: 19.0}  # se 1 é 5% dos dados
# Outras alternativas:
# - Oversampling: SMOTE da imblearn
# - Undersampling: RandomUnderSampler
# - Modelos com built-in support: XGBoost com scale_pos_weight

## Slide 32 — PIPELINE EM CÓDIGO VETORIZAÇÃO + CLASSIFICADOR EM UM OBJETO

*Bloco 07*

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
# Tudo em um único fit/predict
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
# Salvar o pipeline inteiro
import joblib
joblib.dump(pipe, 'sentiment_model.joblib')
# Carregar e usar em produção
pipe = joblib.load('sentiment_model.joblib')
y_pred = pipe.predict(['Adorei o produto!', 'Péssimo atendimento.'])
# array([1, 0])

## Slide 33 — GRIDSEARCHCV NO PIPELINE TUNING DE VETORIZAÇÃO + CLASSIFICADOR JUNTOS

*Bloco 08*

In [ ]:
from sklearn.model_selection import GridSearchCV
# Tune AMBOS os componentes do pipeline
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [3, 5, 10],
    'tfidf__max_df': [0.85, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
    'clf__penalty': ['l1', 'l2'],
}
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,                  # 5-fold cross-validation
    scoring='f1_macro',    # otimiza F1 macro
    n_jobs=-1,             # usa todos os CPUs
)
grid.fit(X_train, y_train)
print(f'Melhores parâmetros: {grid.best_params_}')
print(f'Melhor F1 macro CV: {grid.best_score_:.3f}')
print(f'Test F1 macro: {grid.score(X_test, y_test):.3f}')

## Slide 34 — VALIDAÇÃO CRUZADA ESTRATIFICADA PRESERVANDO PROPORÇÕES DE CLASSE

*Bloco 09*

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X_train, y_train,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1
)
print(f'F1 macro por fold: {scores}')
print(f'Média: {scores.mean():.3f} ± {scores.std():.3f}')
# Em GridSearchCV, basta passar StratifiedKFold no parâmetro cv
# (o default cv=5 já faz isso para classificação)

## Slide 40 — CLASSIFICATION_REPORT MÉTRICAS COMPLETAS EM UMA CHAMADA

*Bloco 10*

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
y_pred = pipe.predict(X_test)
# Report textual completo
print(classification_report(y_test, y_pred,
      target_names=['negativo', 'positivo']))
# Matriz de confusão visual
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu',
            xticklabels=['neg', 'pos'],
            yticklabels=['neg', 'pos'])
plt.xlabel('Previsto'); plt.ylabel('Real')
plt.title('Matriz de Confusão')
plt.show()

## Slide 49 — BIO TAGGING O ESQUEMA PADRÃO PARA ANOTAÇÃO DE NER

*Bloco 11*

In [ ]:
# Exemplo: "Maria Silva trabalha em São Paulo"
# Tokens:    [Maria]   [Silva]   [trabalha]  [em]  [São]    [Paulo]
# Tags BIO:  [B-PER]   [I-PER]   [O]         [O]   [B-LOC]  [I-LOC]
# Como ler:
# - 'Maria' começa entidade do tipo PER (B-PER)
# - 'Silva' continua a mesma entidade PER (I-PER)
# - 'trabalha', 'em' não são entidades (O)
# - 'São' começa entidade do tipo LOC (B-LOC)
# - 'Paulo' continua a mesma entidade LOC (I-LOC)
# Variantes: BIOES (adiciona E=End, S=Single)
# BIO é o padrão por simplicidade

## Slide 50 — NER COM SPACY EM PT-BR PT_CORE_NEWS_LG PRÉ-TREINADO

*Bloco 12*

In [ ]:
import spacy
nlp = spacy.load('pt_core_news_lg')
texto = '''
Ontem comprei um iPhone 15 na Magazine Luiza por R$ 5.999.
A entrega foi feita em São Paulo no dia 20 de outubro pela Fedex.
Atendimento da Apple Brasil foi excelente.
'''
doc = nlp(texto)
print('Entidades encontradas:')
for ent in doc.ents:
    print(f'  {ent.text:<25} | {ent.label_:<6} | '
          f'start={ent.start_char}, end={ent.end_char}')
# Acesso programático: doc.ents é tupla de Span
# Cada Span tem .text, .label_, .start_char, .end_char,
# .start (token), .end (token), .root, etc.

## Slide 51 — SAÍDA DO NER EM PT-BR ENTIDADES RECONHECIDAS PELO SPACY

*Bloco 13*

In [ ]:
# Saída ilustrativa:
# Entidades encontradas:
#   iPhone 15                 | MISC   | start=21,  end=30
#   Magazine Luiza            | ORG    | start=33,  end=47
#   R$ 5.999                  | MONEY  | start=51,  end=59
#   São Paulo                 | LOC    | start=86,  end=95
#   20 de outubro             | DATE   | start=104, end=117
#   Fedex                     | ORG    | start=124, end=129
#   Apple Brasil              | ORG    | start=148, end=160
# Em uma única chamada:
# - 7 entidades extraídas
# - 4 tipos diferentes (MISC, ORG, MONEY, LOC, DATE)
# - Posições de char para destaque visual
# - Pronto para alimentar dashboard ou banco de dados
# Visualizar em Jupyter:
from spacy import displacy
displacy.render(doc, style='ent')

## Slide 52 — AVALIAÇÃO DE NER POR ENTIDADE, NÃO POR TOKEN

*Bloco 14*

In [ ]:
# pip install seqeval
from seqeval.metrics import classification_report as ner_report
# y_true e y_pred são listas de listas de tags BIO
# Exemplo:
# y_true[0] = ['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
# y_pred[0] = ['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC']
# [notebook] exemplo mínimo de tags BIO para rodar a célula
y_true = [['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC'], ['B-ORG', 'O', 'O', 'B-LOC']]
y_pred = [['O', 'O', 'B-PER', 'I-PER', 'O', 'B-LOC'], ['B-ORG', 'O', 'O', 'O']]
print(ner_report(y_true, y_pred))
# Saída ilustrativa:
#               precision    recall  f1-score   support
#          LOC       0.92      0.89      0.90       150
#          ORG       0.85      0.81      0.83       200
#          PER       0.94      0.93      0.93        80
#    micro avg       0.89      0.86      0.87       430
#    macro avg       0.90      0.88      0.89       430

## Slide 56 — PIPELINE COMPLETO TF-IDF + LOGREG + NER EM PARALELO

*Bloco 15*

In [ ]:
# [notebook] split de treino para o GridSearch
reviews_train, labels_train = X_train, y_train
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import spacy
# 1) Modelo de classificação de sentimento
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
# 2) GridSearch para tuning
param_grid = {
    'tfidf__min_df': [3, 5, 10],
    'tfidf__max_df': [0.85, 0.95],
    'clf__C': [0.1, 1.0, 10.0],
}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
grid.fit(reviews_train, labels_train)
print(f'Melhores parâmetros: {grid.best_params_}')
print(f'Melhor F1 macro: {grid.best_score_:.3f}')

## Slide 57 — FUNÇÃO ENRIQUECER_REVIEW SENTIMENTO + ENTIDADES EM UMA CHAMADA

*Bloco 16*

In [ ]:
# 3) NER em paralelo no mesmo texto
nlp = spacy.load('pt_core_news_lg')
def enriquecer_review(texto):
    # Sentiment via pipeline treinado
    sentimento = grid.predict([texto])[0]
    # Entidades via spaCy
    doc = nlp(texto)
    entidades = [(ent.text, ent.label_) for ent in doc.ents]
    return {
        'texto': texto,
        'sentimento': int(sentimento),
        'entidades': entidades
    }
# Uso
resultado = enriquecer_review(
    'Adorei o iPhone 15 da Magazine Luiza, entrega rápida em SP!'
)
print(resultado)
# {'texto': '...', 'sentimento': 1,
#  'entidades': [('iPhone 15', 'MISC'),
#                ('Magazine Luiza', 'ORG'), ('SP', 'LOC')]}